# 任务 5：爬虫/机器人账号检测

## 任务描述
识别电商数据集中的异常账号行为，检测可能的爬虫或机器人账号

## 检测规则
| 规则 | 描述 | 阈值 |
|------|------|------|
| 1. 超高频点击 | 1 分钟内点击次数过多 | > 60 次/分钟 |
| 2. 24 小时无休 | 活跃小时数覆盖全天 | ≥ 23 小时 |
| 3. 单一行为 | 只浏览无深度互动 | PV 比例 > 95% 且无购买/收藏/加购 |

In [ ]:
import polars as pl
import time

print(f"Polars 版本：{pl.__version__}")

In [ ]:
# 配置
INPUT_PATH = "clean_data_deduped.parquet"
THRESHOLD_HIGH_FREQ = 60  # 1 分钟内点击次数阈值

start_time = time.time()

## 1. 加载数据并统计总体信息

In [ ]:
q = pl.scan_parquet(INPUT_PATH)

# 统计总用户数和总请求数
total_stats = q.select(
    pl.col("user_id").n_unique().alias("total_users"),
    pl.len().alias("total_requests")
).collect(engine="streaming")

total_users = total_stats.row(0)[0]
total_requests = total_stats.row(0)[1]

print(f"总用户数：{total_users:,}")
print(f"总请求数：{total_requests:,}")

## 2. 规则 1：超高频点击检测

检测 1 分钟内点击次数超过阈值的用户

In [ ]:
print(f"正在检测超高频点击（阈值：{THRESHOLD_HIGH_FREQ} 次/分钟）...")

# 计算每分钟点击数并找出嫌疑用户
suspect_users_high_freq = (
    q
.with_columns((pl.col("timestamp") // 60).alias("minute_bucket"))
    .group_by("user_id", "minute_bucket")
    .agg(pl.len().alias("clicks_per_minute"))
    .filter(pl.col("clicks_per_minute") > THRESHOLD_HIGH_FREQ)
    .select(pl.col("user_id").unique())
    .collect(engine="streaming")
)

print(f"触发规则 1（超高频点击）的用户数：{suspect_users_high_freq.height:,}")

## 3. 规则 2 & 3：24 小时无休 + 单一行为检测

In [ ]:
print("正在计算用户行为特征...")

# 用户级别特征聚合
user_features = (
    q
    .group_by("user_id")
    .agg(
        pl.len().alias("total_actions"),
        pl.col("behavior_type").filter(pl.col("behavior_type") == "pv").count().alias("pv_count"),
        pl.col("behavior_type").filter(pl.col("behavior_type").is_in(["fav", "cart", "buy"])).count().alias("deep_count"),
        (pl.col("timestamp") * 1_000_000).cast(pl.Datetime("us")).dt.hour().n_unique().alias("active_hours"),
    )
    .collect(engine="streaming")
)

# 规则 2: 24 小时无休
suspect_users_24h = user_features.filter(
    (pl.col("active_hours") >= 23) & (pl.col("total_actions") > 100)
).select(pl.col("user_id"))

# 规则 3: 单一行为（只浏览）
suspect_users_pv_only = user_features.filter(
    (pl.col("pv_count") / pl.col("total_actions").clip(1)) > 0.95
    & (pl.col("deep_count") == 0)
).select(pl.col("user_id"))

print(f"触发规则 2（24 小时无休）的用户数：{suspect_users_24h.height:,}")
print(f"触发规则 3（单一行为）的用户数：{suspect_users_pv_only.height:,}")

## 4. 合并嫌疑用户并统计

In [ ]:
# 合并所有嫌疑用户（去重）
all_suspect_users = pl.concat([
    suspect_users_high_freq,
    suspect_users_24h,
    suspect_users_pv_only
]).unique()

suspect_user_count = all_suspect_users.height
suspect_user_ratio = suspect_user_count / total_users * 100 if total_users > 0 else 0

print(f"\n总嫌疑账号数：{suspect_user_count:,}")
print(f"嫌疑账号占比：{suspect_user_ratio:.2f}%")

## 5. 统计嫌疑账号贡献的请求数

In [ ]:
if suspect_user_count > 0:
    suspect_user_ids = all_suspect_users["user_id"].to_list()
    
    # 统计嫌疑用户的总请求数
    suspect_requests = (
        q
        .filter(pl.col("user_id").is_in(suspect_user_ids))
        .select(pl.len())
        .collect(engine="streaming")
        .item()
    )
else:
    suspect_requests = 0

suspect_request_ratio = suspect_requests / total_requests * 100 if total_requests > 0 else 0

print(f"嫌疑账号贡献请求数：{suspect_requests:,}")
print(f"嫌疑账号请求占比：{suspect_request_ratio:.2f}%")

## 6. 结果汇总

In [ ]:
end_time = time.time()
total_time = end_time - start_time

print("\n" + "=" * 60)
print("爬虫/机器人检测结果")
print("=" * 60)

print(f"""
┌─────────────────────────────────────────────────────────────┐
│                    检测结果汇总                              │
├─────────────────────────────────────────────────────────────┤
│  总用户数：{total_users:>15,}                                    │
│  总请求数：{total_requests:>15,}                                    │
├─────────────────────────────────────────────────────────────┤
│  规则 1 - 超高频点击：{suspect_users_high_freq.height:>10,} 用户                      │
│  规则 2 - 24 小时无休：{suspect_users_24h.height:>10,} 用户                       │
│  规则 3 - 单一行为：{suspect_users_pv_only.height:>10,} 用户                        │
├─────────────────────────────────────────────────────────────┤
│  嫌疑账号总数：{suspect_user_count:>12,}                                  │
│  嫌疑账号占比：{suspect_user_ratio:>12.2f}%                              │
│  嫌疑请求占比：{suspect_request_ratio:>12.2f}%                              │
├─────────────────────────────────────────────────────────────┤
│  检测耗时：{total_time:>14.2f} 秒                                │
└─────────────────────────────────────────────────────────────┘
""")

## 7. 保存结果

In [ ]:
# 保存嫌疑用户列表
all_suspect_users.write_csv("bot_users.csv")
print("嫌疑账号列表已保存至：bot_users.csv")

# 保存汇总指标
summary = pl.DataFrame({
    "metric": ["total_users", "total_requests", "suspect_users", "suspect_requests", "suspect_ratio"],
    "value": [str(total_users), str(total_requests), str(suspect_user_count), str(suspect_requests), f"{suspect_request_ratio:.2f}%"]
})
summary.write_csv("bot_detection_summary.csv")
print("汇总结果已保存至：bot_detection_summary.csv")